## Physical Activity Risk Prediction & Recommandations

### 1) Data Collection
- Dataset Source - https://drive.google.com/file/d/1jrOQdg4fjLljNFnLxAbMouav4s1WzJxa/view?usp=drive_link

### 1.1 Import Data and Required Packages
####  Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [19]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from imblearn.over_sampling import SMOTE

#### Import the CSV Data as Pandas DataFrame

In [20]:
physicalActivity_df = pd.read_csv('../../data/PhysicalActivityParameter.csv')

#### Shape of the Dataset

In [21]:
physicalActivity_df.shape

(697, 15)

#### Head of the Dataset

In [22]:
physicalActivity_df.head()

,Id,Age,Gender,Height,Weight,Energy Levels,PhysicalActivity,SittingTime,Cardiovascular Health,Muscle Strength,Flexibility,Balance,Thirsty,Pain or Discomfort,Available Time
0,1,24,Male,167.0,50.0,2,2,Yes,No,No,No,No,2,No,50
1,2,30,Male,164.0,60.0,1,3,Yes,Yes,No,Yes,Yes,3,No,120
2,3,24,Male,165.0,55.0,3,1,Yes,Yes,No,Yes,Yes,1,Yes,120
3,4,28,Female,153.0,48.0,4,2,Yes,No,Yes,Yes,Yes,2,No,300
4,5,24,Female,163.0,56.0,2,3,Yes,Yes,No,Yes,Yes,5,No,120


### 2.2 Dataset information


- gender : sex of students  -> (Male/female)
- Age : -> 30-50
- Height : -> (cm)
- Weight:-> (kg) 
- EnergyLevels ->1-10
- PhysicalActivity ->1-5
- SittingTime -> Yes/No
- Cardiovascular Health -> Yes/No
- Muscle Strength -> Yes/No
- Flexibility -> Yes/No
- Balance -> Yes/No
- Thirsty -> 1-10
- Pain or Discomfort - Yes/No
- Available Time - Time(minutes)


### 3. Data Checks to perform

- Check Missing values
- Check Duplicates
- Check data type
- Check the number of unique values of each column
- Check statistics of data set
- Check various categories present in the different categorical column

### 3.1 Check Missing values

In [23]:
physicalActivity_df.isna().sum()

Id                       0
Age                      0
Gender                   0
Height                   0
Weight                   0
Energy Levels            0
PhysicalActivity         0
SittingTime              3
Cardiovascular Health    0
Muscle Strength          0
Flexibility              0
Balance                  0
Thirsty                  0
Pain or Discomfort       0
Available Time           0
dtype: int64

#### Convert Categorical colums to numerical

In [24]:
binary_columns = ["SittingTime", "Cardiovascular Health", "Muscle Strength",
                  "Flexibility", "Balance", "Pain or Discomfort", ]
for col in binary_columns:
    physicalActivity_df[col] = physicalActivity_df[col].map({"Yes": 1, "No": 0})

physicalActivity_df["Gender"] = physicalActivity_df["Gender"].map({"Male": 1, "Female": 0})

#### Check duplicates

In [25]:
physicalActivity_df.duplicated().sum()

0

### 3.3 Check data types


In [27]:
physicalActivity_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 697 entries, 0 to 696
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Id                     697 non-null    int64  
 1   Age                    697 non-null    int64  
 2   Gender                 697 non-null    int64  
 3   Height                 697 non-null    float64
 4   Weight                 697 non-null    float64
 5   Energy Levels          697 non-null    int64  
 6   PhysicalActivity       697 non-null    int64  
 7   SittingTime            694 non-null    float64
 8   Cardiovascular Health  697 non-null    int64  
 9   Muscle Strength        697 non-null    int64  
 10  Flexibility            697 non-null    int64  
 11  Balance                697 non-null    int64  
 12  Thirsty                697 non-null    int64  
 13  Pain or Discomfort     697 non-null    int64  
 14  Available Time         697 non-null    int64  
dtypes: flo

### 3.4 Checking the number of unique values of each column


In [28]:
physicalActivity_df.nunique()

Id                       697
Age                       31
Gender                     2
Height                    58
Weight                    89
Energy Levels              5
PhysicalActivity           5
SittingTime                2
Cardiovascular Health      2
Muscle Strength            2
Flexibility                2
Balance                    2
Thirsty                    6
Pain or Discomfort         2
Available Time            71
dtype: int64

#### Calculate BMI

In [29]:
physicalActivity_df["BMI"] = physicalActivity_df["Weight"] / (physicalActivity_df["Height"] / 100) ** 2

#### Assign Value Score and fullFill missing values


In [30]:
import pandas as pd
physicalActivity_df.replace('?', pd.NA, inplace=True)

for col in physicalActivity_df:
    if col not in physicalActivity_df.columns:
        physicalActivity_df[col] = pd.NA  

#### Rename Colums Correctly

In [31]:
physicalActivity_df.rename(columns={
    'Energy Levels	': 'Energy_Levels	',
    'PhysicalActivity': 'Physical_Activity',
    'SittingTime': 'Sitting_Time',
    'Cardiovascular Health': 'Cardiovascular_Health',
    'Muscle Strength': 'Muscle_Strength',
    'Pain or Discomfort':"Pain_or_Discomfort",
    'Available Time': 'Available_Time',
}, inplace=True)

In [32]:
physicalActivity_df.head()

,Id,Age,Gender,Height,Weight,Energy Levels,Physical_Activity,Sitting_Time,Cardiovascular_Health,Muscle_Strength,Flexibility,Balance,Thirsty,Pain_or_Discomfort,Available_Time,BMI
0,1,24,1,167.0,50.0,2,2,1.0,0,0,0,0,2,0,50,17.928215
1,2,30,1,164.0,60.0,1,3,1.0,1,0,1,1,3,0,120,22.308150
2,3,24,1,165.0,55.0,3,1,1.0,1,0,1,1,1,1,120,20.202020
3,4,28,0,153.0,48.0,4,2,1.0,0,1,1,1,2,0,300,20.504934
4,5,24,0,163.0,56.0,2,3,1.0,1,0,1,1,5,0,120,21.077195


### Merged with Diabetic Risk value with Physical Activity Dataset

#### Load the diabetic dataset

In [33]:
diabetic_risk_df=pd.read_csv('../../data/preproccedData/PreProccedWithoutAugmented/PreProccedCommonParameters.csv')

In [34]:
diabetic_risk_df.head()

,Age,Gender,Height,Weight,Waist_Circumference,Diet_Food_Habits,Family_History,Blood_Pressure,Cholesterol_Lipid_Levels,Thirst,Fatigue,Urination,Vision Changes,BMI,DiabetesRisk,RiskLevel
0,24.0,1.0,167.0,50.0,30.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.928215,30.774236,1
1,30.0,1.0,164.0,60.0,32.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.308150,42.784055,1
2,24.0,1.0,165.0,55.0,30.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,20.202020,32.703477,1
3,28.0,0.0,153.0,48.0,28.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.504934,28.756909,0
4,24.0,0.0,163.0,56.0,31.0,4.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,21.077195,37.049640,1


In [35]:
nutrition_df = physicalActivity_df.reset_index(drop=True)
selected_columns = diabetic_risk_df[['DiabetesRisk',]].reset_index(drop=True)

# Now merge
merged_df = pd.concat([physicalActivity_df, selected_columns], axis=1)

# Check the result
print("Merged shape:", merged_df.shape) 

Merged shape: (697, 17)


In [36]:
merged_df.head()

,Id,Age,Gender,Height,Weight,Energy Levels,Physical_Activity,Sitting_Time,Cardiovascular_Health,Muscle_Strength,Flexibility,Balance,Thirsty,Pain_or_Discomfort,Available_Time,BMI,DiabetesRisk
0,1,24,1,167.0,50.0,2,2,1.0,0,0,0,0,2,0,50,17.928215,30.774236
1,2,30,1,164.0,60.0,1,3,1.0,1,0,1,1,3,0,120,22.308150,42.784055
2,3,24,1,165.0,55.0,3,1,1.0,1,0,1,1,1,1,120,20.202020,32.703477
3,4,28,0,153.0,48.0,4,2,1.0,0,1,1,1,2,0,300,20.504934,28.756909
4,5,24,0,163.0,56.0,2,3,1.0,1,0,1,1,5,0,120,21.077195,37.049640
